# 1.1 Build the Cohorts

**Class 1.** A **cohort** is a group of people chosen by a set of rules (for example: aged 25 to 50, with a diagnosis of diabetes). In this notebook we pull our two cohorts out of the All of Us database with **SQL**.

### By the end of this notebook you'll be able to

1. Explain how the All of Us data is organised: the **CDR**, the **OMOP** tables, and **concept IDs**
2. Read a SQL query: what `SELECT`, `FROM`, `JOIN`, `WHERE`, and `IN` / `NOT IN` do
3. Define a **study group** and a **control group** with inclusion and exclusion criteria
4. Run a query from R and save the result to your bucket

| | |
|---|---|
| ⏱️ **Time** | 60–75 minutes (the queries take a few minutes each) |
| 📂 **Uses** | the All of Us dataset (BigQuery) |
| 💾 **Saves** | `person_study.csv`, `person_control.csv`, `survey_study.csv`, `survey_control.csv`, `measurement_study.csv`, `measurement_control.csv` |

**Before you start:**

- Run **0.0 setup**, so `aou_setup.R` has your bucket and dataset

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data, or counts of 1 to 20. Clear outputs before you share or commit this notebook (**Edit → Clear Outputs of All Cells**).

## Our two cohorts

| | Study group | Control group |
|---|---|---|
| Age | 25 to 50 | 25 to 50 |
| Alive | ✓ | ✓ |
| Race | White, Black, or Asian | same |
| Ethnicity | Hispanic or not Hispanic | same |
| Gender | Male or Female | same |
| **Diabetes diagnosis** | **Yes** (included) | **No** (excluded) |

The two groups follow **exactly the same rules except one**: whether the person has a diabetes diagnosis. That's what makes them comparable. Any differences we find later are differences between people **with** and **without** diabetes.

For each group we pull three tables:

| Table | Variables | Saved as |
|---|---|---|
| Demographics | date of birth, gender, race, ethnicity | `person_study.csv` / `person_control.csv` |
| Survey | two PTSD questions: *still seeing a doctor?* and *currently receiving treatment?* | `survey_study.csv` / `survey_control.csv` |
| Measurements | BMI, systolic blood pressure, LDL cholesterol, HbA1c | `measurement_study.csv` / `measurement_control.csv` |

Each group is saved to **its own file**. In notebook **2.0** we flag which group each person belongs to and bring them together.

## Key concepts

### The CDR and the OMOP tables

The All of Us data is stored in BigQuery as the **Curated Data Repository (CDR)**: a set of tables, released in numbered **versions** (e.g. CDR v9). The tables follow a common standard called the **OMOP Common Data Model**, so data from many hospitals and surveys fits the same structure. The tables we use:

| Table | One row per… | Example columns |
|---|---|---|
| `person` | participant | `person_id`, `birth_datetime`, `gender_concept_id`, `race_concept_id` |
| `condition_occurrence` | diagnosis | `person_id`, `condition_concept_id`, `condition_start_datetime` |
| `measurement` | lab result or vital sign | `person_id`, `measurement_concept_id`, `value_as_number` |
| `concept` | code in the dictionary | `concept_id`, `concept_name` |
| `death` | participant who has died | `person_id`, `death_date` |

Every table shares **`person_id`**, which is how we link a person's data across tables.

### Concept IDs

Instead of storing text like *"Type 2 diabetes"*, OMOP tables store a number, a **concept ID**, that points to an entry in the `concept` table. The same idea always has the same ID, no matter which hospital recorded it. The IDs used in our queries:

| Concept ID | Means |
|---|---|
| 201820 | Diabetes mellitus (and, through the hierarchy, every more specific type) |
| 8527, 8516, 8515 | White, Black or African American, Asian |
| 38003563, 38003564 | Hispanic or Latino, Not Hispanic or Latino |
| 45878463, 45880669 | Female, Male |
| 43528846, 43530361 | the two PTSD survey questions |

You can look up any concept ID at [athena.ohdsi.org](https://athena.ohdsi.org/).

### Cohort Builder tables

Our queries were **generated by the Cohort Builder** in the classic Workbench. Besides the OMOP tables, they use a few helper tables it relies on: `cb_search_person` (one row per person, ready for filtering), `cb_criteria` (the concept hierarchy), `cb_search_all_events` (every coded event), and `ds_survey` (survey answers in an easy-to-read form).

In [ ]:
source("../aou_setup.R")  # settings, packages, and helper functions

## How we updated the queries for Workbench 2.0

We changed only two things in the Cohort Builder's SQL:

1. **The dataset path.** Every table is written as `` `{CURRENT_DATASET}.table` ``, so it points at the dataset in `aou_setup.R`:

   | | |
   |---|---|
   | Classic Workbench ❌ | `` FROM `fc-aou-cdr-prod-ct.C2024Q3R8.person` `` |
   | Workbench 2.0 ✅ | `` FROM `{CURRENT_DATASET}.person` `` |

2. **One word for the two groups.** The study and control queries were identical except for **one word**: the study group **includes** people with diabetes (`IN`), and the control group **excludes** them (`NOT IN`). We replaced that word with `{diabetes_filter}`, so each query is written once and run twice: with `"IN"` for the study group and `"NOT IN"` for the control group.

`str_glue()` fills in `{CURRENT_DATASET}` and `{diabetes_filter}` just before the query runs.

## Check that the tables exist

Not every dataset has the Cohort Builder tables. This asks BigQuery for the list of tables in your dataset (`INFORMATION_SCHEMA.TABLES` is a built-in list) and checks the ones we need.

In [ ]:
needed <- c("person", "concept", "death", "measurement", "ds_survey",
            "cb_search_person", "cb_criteria", "cb_search_all_events")

found <- run_query(str_glue(
  "SELECT table_name FROM `{CURRENT_DATASET}`.INFORMATION_SCHEMA.TABLES"))$table_name

tibble(table = needed, found = needed %in% found)

**🔍 Read the output:** every row should say `TRUE`. If any say `FALSE`, this dataset doesn't have those tables; use the optional **Data Explorer** route at the end of this notebook instead.

`%in%` asks "is each item on the left found in the list on the right?" and answers `TRUE` or `FALSE` for each.

## A quick guide to reading SQL

SQL describes **what** data you want; BigQuery works out how to get it. The main parts:

| Part | Means | Example |
|---|---|---|
| `SELECT` | which columns to return | `SELECT person.person_id, person.birth_datetime` |
| `FROM` | which table | `` FROM `{CURRENT_DATASET}.person` person `` (the second `person` is a short nickname, an **alias**) |
| `LEFT JOIN … ON` | add columns from another table where IDs match | join `concept` to turn `gender_concept_id` into a readable name |
| `WHERE` | keep only rows that meet a condition | `WHERE race_concept_id IN (8527, 8516, 8515)` |
| `IN (…)` / `NOT IN (…)` | is the value in this list? | `person_id IN (SELECT …)` |
| `(SELECT …)` | a **subquery**: a query inside a query that produces a list | the list of everyone with a diabetes diagnosis |
| `as` | rename a column in the result | `concept_name as gender` |

The demographics query below has three parts:

1. **`SELECT … FROM person LEFT JOIN concept …`:** the columns we want, with concept IDs turned into names
2. **`WHERE person.PERSON_ID IN (SELECT … FROM cb_search_person …)`:** keep only people in our cohort. Inside it, each `AND cb_search_person.person_id IN (…)` is one rule: age 25–50 and alive, ethnicity, gender, race
3. **The last rule**, `cb_search_person.person_id {diabetes_filter} (SELECT criteria.person_id …)`: the list of everyone with a diagnosis under concept **201820** (diabetes). With `IN` we keep them (study group); with `NOT IN` we leave them out (control group)

> 💡 The SQL is written in R as a **raw string**, `r"---( … )---"`. Everything between the brackets is kept exactly as typed, so the SQL can be pasted in as the Cohort Builder wrote it.

## Query 1: Demographics

In [ ]:
person_sql <- r"---(
      SELECT
        person.person_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        p_race_concept.concept_name as race,
        p_ethnicity_concept.concept_name as ethnicity,
        p_sex_at_birth_concept.concept_name as sex_at_birth 
    FROM
        `{CURRENT_DATASET}.person` person 
    LEFT JOIN
        `{CURRENT_DATASET}.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `{CURRENT_DATASET}.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `{CURRENT_DATASET}.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `{CURRENT_DATASET}.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `{CURRENT_DATASET}.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `{CURRENT_DATASET}.cb_search_person` p 
            WHERE
                DATE_DIFF(CURRENT_DATE, dob, YEAR) - IF(EXTRACT(MONTH FROM dob)*100 + EXTRACT(DAY FROM dob) > EXTRACT(MONTH FROM CURRENT_DATE)*100 + EXTRACT(DAY FROM CURRENT_DATE), 1, 0) BETWEEN 25 AND 50 
                AND NOT EXISTS (      SELECT
                    'x'      
                FROM
                    `{CURRENT_DATASET}.death` d      
                WHERE
                    d.person_id = p.person_id ) ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `{CURRENT_DATASET}.person` p 
            WHERE
                ethnicity_concept_id IN (38003564, 38003563) ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `{CURRENT_DATASET}.person` p 
            WHERE
                gender_concept_id IN (45878463, 45880669) ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `{CURRENT_DATASET}.person` p 
            WHERE
                race_concept_id IN (8527, 8516, 8515) ) 
            AND cb_search_person.person_id {diabetes_filter} (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `{CURRENT_DATASET}.cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `{CURRENT_DATASET}.cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `{CURRENT_DATASET}.cb_criteria` cr       
                        WHERE
                            concept_id IN (201820)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) 
                    AND is_standard = 1 )) criteria ) )
)---"

person_study   <- run_query(str_glue(person_sql, diabetes_filter = "IN"))      # has diabetes
person_control <- run_query(str_glue(person_sql, diabetes_filter = "NOT IN"))  # no diabetes

glimpse(person_study)

**🔍 Read the output:** `glimpse()` shows the number of rows (people) and columns, then each column with its type and first few values:

- `<dbl>` a number, `<chr>` text, `<dttm>` a date and time
- `date_of_birth` is a date-time; we'll turn it into an age in notebook 2.0

The first time, this cell takes a minute or two: it runs the query twice (once per group).

**📦 Functions used**

| Function | Package | What it does |
|---|---|---|
| `r"---(…)---"` | base R | a raw string: text kept exactly as typed |
| `str_glue()` | stringr | fills `{CURRENT_DATASET}` and `{diabetes_filter}` into the SQL |
| `run_query()` | our helper (`aou_setup.R`) | sends SQL to BigQuery, returns a data frame |
| `glimpse()` | dplyr | a quick look at a data frame's columns |

Save both groups to your bucket. You won't need to run the query again.

In [ ]:
save_data(person_study,   "person_study.csv")
save_data(person_control, "person_control.csv")

### ✍️ Your turn

1. How many people are in each group? (Hint: `nrow()` counts the rows of a data frame.)
2. Which group is bigger, and does that make sense?

<details>
<summary><b>Show answer</b></summary>

```r
nrow(person_study)
nrow(person_control)
```

The **control** group is much bigger: most adults aged 25 to 50 don't have diabetes. Keep these numbers to yourself: group sizes are fine to look at, but follow the dissemination rules before sharing any counts.

</details>

In [ ]:
# Your code here

## Query 2: Survey answers

Survey answers come from the `ds_survey` table. We keep only the two PTSD questions (concepts **43528846** and **43530361**). The result is **long**: one row per person *per question* they answered.

In [ ]:
survey_sql <- r"---(
    SELECT
        answer.person_id,
        answer.question,
        answer.answer  
    FROM
        `{CURRENT_DATASET}.ds_survey` answer   
    WHERE
        (
            question_concept_id IN (43528846, 43530361)
        )  
        AND (
            answer.PERSON_ID IN (SELECT
                distinct person_id  
            FROM
                `{CURRENT_DATASET}.cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `{CURRENT_DATASET}.cb_search_person` p 
                WHERE
                    DATE_DIFF(CURRENT_DATE, dob, YEAR) - IF(EXTRACT(MONTH FROM dob)*100 + EXTRACT(DAY FROM dob) > EXTRACT(MONTH FROM CURRENT_DATE)*100 + EXTRACT(DAY FROM CURRENT_DATE), 1, 0) BETWEEN 25 AND 50 
                    AND NOT EXISTS (      SELECT
                        'x'      
                    FROM
                        `{CURRENT_DATASET}.death` d      
                    WHERE
                        d.person_id = p.person_id ) ) 
                AND cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `{CURRENT_DATASET}.person` p 
                WHERE
                    ethnicity_concept_id IN (38003564, 38003563) ) 
                AND cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `{CURRENT_DATASET}.person` p 
                WHERE
                    gender_concept_id IN (45878463, 45880669) ) 
                AND cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `{CURRENT_DATASET}.person` p 
                WHERE
                    race_concept_id IN (8527, 8516, 8515) ) 
                AND cb_search_person.person_id {diabetes_filter} (SELECT
                    criteria.person_id 
                FROM
                    (SELECT
                        DISTINCT person_id, entry_date, concept_id 
                    FROM
                        `{CURRENT_DATASET}.cb_search_all_events` 
                    WHERE
                        (concept_id IN(SELECT
                            DISTINCT c.concept_id 
                        FROM
                            `{CURRENT_DATASET}.cb_criteria` c 
                        JOIN
                            (SELECT
                                CAST(cr.id as string) AS id       
                            FROM
                                `{CURRENT_DATASET}.cb_criteria` cr       
                            WHERE
                                concept_id IN (201820)       
                                AND full_text LIKE '%_rank1]%'      ) a 
                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                OR c.path LIKE CONCAT('%.', a.id) 
                                OR c.path LIKE CONCAT(a.id, '.%') 
                                OR c.path = a.id) 
                        WHERE
                            is_standard = 1 
                            AND is_selectable = 1) 
                        AND is_standard = 1 )) criteria ) )
        )
)---"

survey_study   <- run_query(str_glue(survey_sql, diabetes_filter = "IN"))      # has diabetes
survey_control <- run_query(str_glue(survey_sql, diabetes_filter = "NOT IN"))  # no diabetes

glimpse(survey_study)

Look at the exact wording of the questions and answers. Notebook 2.0 relies on it to tidy the answers. `distinct()` lists each unique combination once.

In [ ]:
survey_study %>% distinct(question, answer)

**🔍 Read the output:** each question appears with several answers: *Yes*, *No*, and others like *Skip* or *Prefer not to answer*. In 2.0 we keep only *Yes* and *No*.

In [ ]:
save_data(survey_study,   "survey_study.csv")
save_data(survey_control, "survey_control.csv")

## Query 3: Measurements

BMI, systolic blood pressure, LDL cholesterol, and HbA1c. The query finds them through the concept hierarchy (`cb_criteria`), so every more specific version of each measurement is included. That's why you'll see many slightly different names in 2.0.

This is the **biggest** query: **millions of rows**, one per measurement. Expect it to take **several minutes**. Leave it running (`[*]`) and don't run it again, or it starts over.

In [ ]:
measurement_sql <- r"---(
      SELECT
        measurement.person_id,
        measurement.measurement_concept_id,
        m_standard_concept.concept_name as standard_concept_name,
        measurement.measurement_datetime,
        measurement.value_as_number 
    FROM
        ( SELECT
            * 
        FROM
            `{CURRENT_DATASET}.measurement` measurement 
        WHERE
            (
                measurement_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `{CURRENT_DATASET}.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `{CURRENT_DATASET}.cb_criteria` cr       
                    WHERE
                        concept_id IN (3004249, 3004410, 3038553, 40795800, 4152194)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                measurement.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `{CURRENT_DATASET}.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `{CURRENT_DATASET}.cb_search_person` p 
                    WHERE
                        DATE_DIFF(CURRENT_DATE, dob, YEAR) - IF(EXTRACT(MONTH FROM dob)*100 + EXTRACT(DAY FROM dob) > EXTRACT(MONTH FROM CURRENT_DATE)*100 + EXTRACT(DAY FROM CURRENT_DATE), 1, 0) BETWEEN 25 AND 50 
                        AND NOT EXISTS (      SELECT
                            'x'      
                        FROM
                            `{CURRENT_DATASET}.death` d      
                        WHERE
                            d.person_id = p.person_id ) ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `{CURRENT_DATASET}.person` p 
                    WHERE
                        ethnicity_concept_id IN (38003564, 38003563) ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `{CURRENT_DATASET}.person` p 
                    WHERE
                        gender_concept_id IN (45878463, 45880669) ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `{CURRENT_DATASET}.person` p 
                    WHERE
                        race_concept_id IN (8527, 8516, 8515) ) 
                    AND cb_search_person.person_id {diabetes_filter} (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `{CURRENT_DATASET}.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT c.concept_id 
                            FROM
                                `{CURRENT_DATASET}.cb_criteria` c 
                            JOIN
                                (SELECT
                                    CAST(cr.id as string) AS id       
                                FROM
                                    `{CURRENT_DATASET}.cb_criteria` cr       
                                WHERE
                                    concept_id IN (201820)       
                                    AND full_text LIKE '%_rank1]%'      ) a 
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                    OR c.path LIKE CONCAT('%.', a.id) 
                                    OR c.path LIKE CONCAT(a.id, '.%') 
                                    OR c.path = a.id) 
                            WHERE
                                is_standard = 1 
                                AND is_selectable = 1) 
                            AND is_standard = 1 )) criteria ) )
            )) measurement 
    LEFT JOIN
        `{CURRENT_DATASET}.concept` m_standard_concept 
            ON measurement.measurement_concept_id = m_standard_concept.concept_id
)---"

measurement_study   <- run_query(str_glue(measurement_sql, diabetes_filter = "IN"))      # has diabetes
measurement_control <- run_query(str_glue(measurement_sql, diabetes_filter = "NOT IN"))  # no diabetes

glimpse(measurement_study)

In [ ]:
save_data(measurement_study,   "measurement_study.csv")
save_data(measurement_control, "measurement_control.csv")

## Check what you saved

These files are now in your bucket's `data/` folder. `list.files()` lists the files in a folder.

In [ ]:
list.files(DATA_DIR)

### ✍️ Your turn

Imagine you wanted to study **hypertension** instead of diabetes. Hypertension's concept ID is **316866**. Which part of the queries would you change?

<details>
<summary><b>Show answer</b></summary>

The **last rule** in each query, inside the subquery that builds the diabetes list:

```sql
concept_id IN (201820)
```

You'd change `201820` to `316866`. Because the rule uses the concept **hierarchy**, this picks up every more specific type of hypertension too. Everything else (age, race, ethnicity, gender) stays the same.

In practice it's easiest to rebuild the cohort in **Data Explorer** and use the SQL it generates.

</details>

## ⚠️ Common errors

| Error message | What it means | Fix |
|---|---|---|
| *Not found: Table … cb_search_person* | This dataset doesn't have the Cohort Builder tables | Use the Data Explorer route below |
| *Access Denied* | Wrong `CURRENT_DATASET`, or no Controlled Tier access | Check `aou_setup.R` against a Data Explorer query in this workspace |
| The app freezes or restarts during the measurement query | It ran out of memory | Stop other apps, or edit the app to use a machine with more memory |
| *Can't find your bucket folder* | `BUCKET_NAME` is wrong | Rerun **0.0 setup** |

## ✅ Key takeaways

- The All of Us data follows the **OMOP** model: tables linked by `person_id`, with **concept IDs** for everything coded
- A **cohort** is defined by inclusion and exclusion rules; our study and control groups differ by **one** rule
- SQL `WHERE … IN (…)` keeps rows in a list; `NOT IN` leaves them out
- Run queries **once** and save the results: queries cost money and time

**You're done with 1.1.** Next: **1.2 survival_data**. (The section below is optional.)

---

## Optional: use a Data Explorer export instead

> ⏭️ **Ran the SQL above? Skip this section and go to 1.2.** Your data is already saved.
>
> This section is **only** for people who build their cohorts in **Data Explorer** instead of running the SQL. The cells below have placeholder paths (`EDIT/...`) that you'd fill in first.

You can build the same two groups with **Data Explorer**, the point-and-click tool in Workbench 2.0. **Step-by-step instructions for both cohorts are in the notebooks README:** [notebooks/README.md](https://github.com/CWML/All-of-Us-Training-Resources/tree/main/notebooks#build-the-cohorts-in-data-explorer). In short:

1. Add **two cohorts**, using the criteria at the top of this notebook: one **with** diabetes (Condition: *Diabetes mellitus*) and one **without** (exclude it)
2. For each cohort, **Save data snapshot** with *Demographics*, the two PTSD survey questions, and the four measurements, as an **R notebook**
3. Run each exported notebook once. It saves CSV files to your bucket under `bq_exports/`
4. Then use the code below to read them and save them with the **same file names and columns** as above, so notebooks 2.0 to 3.4 work unchanged

The exported **person** table uses different column names (e.g. `T_DISP_gender`). This renames them:

In [ ]:
# Find the exported files (they're in folders named by table and date)
list.files(file.path(BUCKET_DIR, "bq_exports"), recursive = TRUE, pattern = "\\.csv$")

In [ ]:
# Read every CSV in one export folder into a single data frame
read_export <- function(folder) {
  list.files(folder, pattern = "\\.csv$", full.names = TRUE) %>%
    map_dfr(read_csv, show_col_types = FALSE)
}

# EDIT these paths: one person export folder per cohort
person_study   <- read_export(file.path(BUCKET_DIR, "bq_exports", "EDIT/study/person/folder"))
person_control <- read_export(file.path(BUCKET_DIR, "bq_exports", "EDIT/control/person/folder"))

# Rename the export's columns to match the SQL version
tidy_person <- function(df) {
  df %>% transmute(person_id,
                   date_of_birth = birth_datetime,
                   gender    = T_DISP_gender,
                   race      = T_DISP_race,
                   ethnicity = T_DISP_ethnicity)
}
person_study   <- tidy_person(person_study)
person_control <- tidy_person(person_control)

# save_data(person_study,   "person_study.csv")     # uncomment to save
# save_data(person_control, "person_control.csv")

Do the same for the **survey** and **measurement** exports: read each cohort, rename columns to match the SQL versions above, and save as `survey_study.csv`, `survey_control.csv`, `measurement_study.csv`, and `measurement_control.csv` (`person_id`, `question`, `answer`; and `person_id`, `measurement_concept_id`, `standard_concept_name`, `measurement_datetime`, `value_as_number`). Use `names()` to see what the export calls them.